# RAG Concepts — Local First with ChromaDB

**No credentials. No cloud costs. Runs 100% locally on your laptop.**

Learn every core concept of Retrieval-Augmented Generation (RAG) step-by-step:

```
Document Text ──► [1] Chunk ──► [2] Embed ──► [3] Store in ChromaDB
                                                      │
User Question ──► [4] Embed ──► [5] Similarity Search ─┘
                                      │
                             [6] Augment Prompt ──► [7] LLM ──► Answer
```

## VS Code / Jupyter Setup
1. Open this `.ipynb` file in VS Code or JupyterLab
2. Pick any Python 3.9+ kernel
3. Run cells one at a time with **Shift+Enter**


In [ ]:
# ── Install everything needed ──────────────────────────────────────────────
# This installs into your current Python environment.
# sentence-transformers downloads a lightweight ~90MB model the first time.

%pip install chromadb sentence-transformers langchain-text-splitters numpy openai python-dotenv


---
## Step 1 — Load Document and Split into Chunks

**Why chunk at all?**

Embedding models have a token limit (~512 tokens). A long document won't fit.
More importantly: smaller, focused chunks match specific questions much better
than a large document where the relevant answer is buried in hundreds of sentences.

```
"Full Document (2000 chars)"
 ├── Chunk 0:  chars   0 → 400
 ├── Chunk 1:  chars 350 → 750   ← 50-char overlap keeps context at boundaries
 ├── Chunk 2:  chars 700 → 1100
 └── ...
```


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Sample Document Text ──────────────────────────────────────────────────
DOCUMENT_TEXT = """The Exploration of Mars: The Red Planet, Water, and Robotic Missions

Mars is the fourth planet from the Sun and the second-smallest planet in the Solar System. It is frequently referred to as the "Red Planet" because the iron oxide (rust) prevalent on its surface gives it a distinct reddish appearance. Mars possesses a very thin atmosphere composed of approximately 95% carbon dioxide, 2.6% molecular nitrogen, 1.9% argon, and trace amounts of water vapor and oxygen. Due to this thin atmosphere, atmospheric pressure at the Martian surface averages less than 1% of Earth's pressure, making it impossible for liquid water to exist stably on the surface for extended periods.

Despite its dry conditions today, evidence gathered by space probes shows that ancient Mars was once much warmer and wetter. Massive outflow channels, dried lake basins, and river valley networks carved across the landscape indicate that liquid water once flowed freely billions of years ago. Today, water on Mars exists primarily in the form of ice. Vast ice deposits are concentrated in the permanent polar ice caps and trapped beneath the shallow mantle of Martian soil. Radar sounding instruments aboard orbiting spacecraft have also detected potential deposits of subglacial liquid water deep beneath the southern polar ice sheet.

Robotic explorers have revolutionized our understanding of Mars. In 2012, NASA landed the Curiosity rover in Gale Crater to investigate whether Mars could ever have supported microbial life. Curiosity discovered chemical evidence of ancient freshwater lakes with the essential elements required for life, including carbon, nitrogen, and oxygen. In February 2021, the Perseverance rover touched down in Jezero Crater, an ancient river delta. Perseverance is equipped to drill and collect Martian rock and soil cores for eventual return to Earth. Accompanying Perseverance was Ingenuity, a solar-powered helicopter drone that achieved the historic first powered, controlled aerodynamic flight on another planet.

The geography of Mars features some of the most extreme geological formations in the Solar System. Olympus Mons is an enormous shield volcano rising roughly 22 kilometers (nearly 72,000 feet) high, making it nearly three times taller than Mount Everest. Another prominent feature is Valles Marineris, an immense system of canyons stretching over 4,000 kilometers along the Martian equator, which is more than four times deeper and ten times longer than the Grand Canyon. Mars also has two small, irregularly shaped moons named Phobos and Deimos, which scientists believe are captured asteroids from the nearby asteroid belt.

Planning crewed expeditions to Mars presents severe engineering and biological challenges. Astronauts traveling to Mars face transit times between six and nine months each way, exposing them to high levels of cosmic radiation and prolonged microgravity that causes bone density loss. On the surface, astronauts must endure frigid temperatures averaging minus 60 degrees Celsius (-80 degrees Fahrenheit) and violent global dust storms that can shroud the entire planet for months at a time, completely blocking sunlight needed for solar arrays. Developing reliable life-support systems, radiation shielding, and in-situ resource utilization for producing fuel and water will be essential for human survival."""

# ── Split into chunks ──────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # max characters per chunk
    chunk_overlap=50,    # overlap so sentence boundaries don't lose context
)
chunks = splitter.split_text(DOCUMENT_TEXT)

# ── See the results ────────────────────────────────────────────────────────
print(f"Total chunks created: {len(chunks)}")
print("=" * 60)
for i, chunk in enumerate(chunks):
    print(f"\n[Chunk {i}] ({len(chunk)} chars)")
    print(chunk.strip())


---
## Step 2 — Generate Embeddings

**What is an embedding?**

An embedding model reads text and outputs a vector (a list of floating-point numbers, e.g. 384 numbers).
Sentences with similar meanings produce vectors that point in almost the same mathematical direction.

```
"Why is Mars red?"         →  [0.021, -0.134, 0.089, 0.241, ...] (384 numbers)
"Iron oxide colors Mars"   →  [0.019, -0.129, 0.091, 0.238, ...] ← very similar!
"How to make pasta dough"  →  [-0.201, 0.441, -0.312, 0.051, ...] ← very different
```

We use `all-MiniLM-L6-v2` — a free, fast model that runs locally on your CPU with no API key.


In [ ]:
%pip install --upgrade --force-reinstall packaging


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# ── Load the embedding model (downloads once, ~90MB) ──────────────────────
print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Model ready")


In [ ]:
# ── See what an embedding actually looks like ──────────────────────────────
sample_text = "Iron oxide on the surface of Mars produces a reddish color."
sample_vector = embed_model.encode(sample_text)

print(f"Text:           \"{sample_text}\"")
print(f"Vector length:  {len(sample_vector)} numbers")
print(f"First 10 values: {np.round(sample_vector[:10], 4).tolist()}")
print(f"Value range:    min={sample_vector.min():.4f}, max={sample_vector.max():.4f}")


In [ ]:
# ── See how similarity works — cosine similarity ───────────────────────────
# Cosine similarity = 1.0 means identical direction (very similar)
#                   = 0.0 means perpendicular (unrelated)
#                   = -1.0 means opposite

def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

sentences = [
    "Why is Mars called the Red Planet?",                          # Target question
    "Iron oxide on the Martian surface gives it a reddish color.",  # Highly relevant chunk
    "Valles Marineris is an immense system of canyons.",           # Related topic (Mars geography)
    "Espresso machines brew coffee under high pressure.",          # Completely unrelated topic
    "Curiosity discovered evidence of ancient freshwater lakes."    # Different Mars topic (rovers)
]

question = sentences[0]
q_vec = embed_model.encode(question)

print(f"Reference question: \"{question}\"")
print("=" * 65)

for s in sentences[1:]:
    s_vec = embed_model.encode(s)
    score = cosine_similarity(q_vec, s_vec)
    bar = "█" * int(max(0, score) * 30)
    print(f"  {score:.4f}  {bar}")
    print(f"         \"{s[:60]}\"")
    print()


In [ ]:
# ── Embed all document chunks ──────────────────────────────────────────────
print(f"Embedding {len(chunks)} chunks...")
chunk_embeddings = embed_model.encode(chunks, show_progress_bar=True)

print(f"\n✅ Done. Shape: {chunk_embeddings.shape}")
print(f"   → {chunk_embeddings.shape[0]} chunks, each with {chunk_embeddings.shape[1]} numbers")


---
## Step 3 — Store in ChromaDB

ChromaDB is a lightweight vector database that runs entirely in-memory (or on local disk).
It stores your text chunks, metadata, and embedding vectors together.
When you search, it computes cosine similarity against every stored vector and returns the closest matches.

```
ChromaDB Collection: "mars_knowledge_base"
┌────────────────────────────────────────────────────────────────────────┐
│  ID      │  Text (chunk)              │  Metadata       │  Vector      │
│──────────┼────────────────────────────┼─────────────────┼──────────────│
│ chunk_0  │  "Mars is the fourth..."   │  {doc: 'mars'}  │ [0.02, ...]  │
│ chunk_1  │  "Despite its dry..."      │  {doc: 'mars'}  │ [-0.13, ...] │
│  ...     │  ...                       │  ...            │  ...         │
└────────────────────────────────────────────────────────────────────────┘
```


In [ ]:
import chromadb

# ── Create an in-memory ChromaDB client ───────────────────────────────────
# EphemeralClient = lives in RAM, gone when Python restarts
# Use PersistentClient(path="./chroma_db") to save to disk

chroma_client = chromadb.EphemeralClient()

# Delete if it already exists (so we can re-run cleanly)
try:
    chroma_client.delete_collection("mars_knowledge_base")
except:
    pass

collection = chroma_client.create_collection(
    name="mars_knowledge_base",
    metadata={"hnsw:space": "cosine"}  # use cosine similarity for search
)

print("✅ ChromaDB collection created: mars_knowledge_base")


In [ ]:
# ── Add all chunks to ChromaDB ─────────────────────────────────────────────
# ChromaDB needs: ids (unique), documents (text), embeddings (vectors), metadatas

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),   # convert numpy array → list
    metadatas=[{"doc_id": "mars_doc", "chunk_index": i} for i in range(len(chunks))]
)

print(f"✅ Indexed {collection.count()} chunks in ChromaDB")

# ── Peek inside the collection ────────────────────────────────────────────
peek = collection.peek(limit=2)
print("\n--- First 2 stored items ---")
for i in range(len(peek["ids"])):
    print(f"\nID:       {peek['ids'][i]}")
    print(f"Metadata: {peek['metadatas'][i]}")
    print(f"Text:     {peek['documents'][i][:100]}...")
    print(f"Vector:   [{", ".join(f"{v:.4f}" for v in peek['embeddings'][i][:5])}, ...]")


---
## Step 4 + 5 — Ask a Question and Search

**How similarity search works:**
1. Embed the user's question → get a question vector
2. Compute cosine similarity between the question vector and every stored chunk vector
3. Return the top-K chunks with the highest similarity scores

This is the "Retrieval" part of RAG — finding the relevant evidence.


In [ ]:
# ── Ask a question ─────────────────────────────────────────────────────────
# Try different questions and see which chunks get retrieved!

USER_QUESTION = "Why is Mars called the Red Planet, and what is its atmosphere made of?"
# USER_QUESTION = "Where can water be found on Mars?"
# USER_QUESTION = "What did the Curiosity rover discover?"
# USER_QUESTION = "What is Olympus Mons?"

K = 3   # how many chunks to return

# Step 4: Embed the question
question_vector = embed_model.encode(USER_QUESTION).tolist()

print(f"Question: \"{USER_QUESTION}\"")
print(f"Question vector (first 8 values): {[round(v, 4) for v in question_vector[:8]]}")
print(f"\nSearching for top {K} most similar chunks...")


In [ ]:
# Step 5: Similarity search in ChromaDB
results = collection.query(
    query_embeddings=[question_vector],
    n_results=K,
    include=["documents", "metadatas", "distances"]
)

# ChromaDB returns 'distances' — lower = more similar (it uses 1 - cosine_similarity)
# Convert to similarity score: similarity = 1 - distance

print("=" * 60)
print(f"Top {K} results:")
print("=" * 60)

retrieved_texts = []

for i in range(K):
    distance  = results["distances"][0][i]
    similarity = 1 - distance          # convert back to 0→1 score
    text      = results["documents"][0][i]
    meta      = results["metadatas"][0][i]

    retrieved_texts.append(text)        # save for prompt in Step 6

    bar = "█" * int(max(0, similarity) * 30)
    print(f"\n[Match {i+1}]  Similarity: {similarity:.4f}  {bar}")
    print(f"Metadata: {meta}")
    print(f"Text: {text.strip()}")


---
## Step 6 — Build the Augmented Prompt

**What is prompt augmentation?**
We inject the retrieved chunks into the LLM prompt as authoritative "context".
This grounds the LLM's answer in your specific document content instead of generic assumptions or outdated knowledge.

Without RAG: The LLM answers from memory → may hallucinate or give generic answers.
With RAG: The LLM sees the exact relevant text → specific, accurate, traceable answers.


In [ ]:
# ── Build context from retrieved chunks ───────────────────────────────────
context = "\n\n".join(
    f"[Excerpt {i+1}]\n{text}"
    for i, text in enumerate(retrieved_texts)
)

# ── Build the augmented prompt ─────────────────────────────────────────────
SYSTEM_PROMPT = """You are a helpful science assistant. Answer the user's question 
using ONLY the context excerpts provided. Be concise, direct, and factual.
If the answer is not in the context, say: 'I cannot find the answer to this question in the provided context.'"""

USER_PROMPT = f"""Context:

{context}

---
Question: {USER_QUESTION}

Answer:"""

print("=" * 60)
print("SYSTEM PROMPT:")
print(SYSTEM_PROMPT)
print("\n" + "=" * 60)
print("USER PROMPT (what goes to the LLM):")
print(USER_PROMPT)


---
## Step 7 — Ask the LLM

**Option A — OpenAI API (optional)**
Add `OPENAI_API_KEY=sk-...` to a `.env` file in this folder.

**Option B — No API key? Paste it yourself for free!**
The cell below will print the full augmented prompt. Copy and paste it into [claude.ai](https://claude.ai) or [chatgpt.com](https://chatgpt.com).


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")

if openai_key:
    # ── Option A: Call OpenAI ─────────────────────────────────────────────
    from openai import OpenAI

    client = OpenAI(api_key=openai_key)

    response = client.chat.completions.create(
        model="gpt-4o-mini",     # cost-efficient, fast model
        temperature=0,           # 0 = deterministic, factual answers
        max_tokens=400,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": USER_PROMPT}
        ]
    )

    answer = response.choices[0].message.content
    tokens = response.usage.total_tokens

    print("=" * 60)
    print(f"Q: {USER_QUESTION}")
    print("=" * 60)
    print(f"A: {answer}")
    print(f"\n(Tokens used: {tokens} — approx ${tokens * 0.00000015:.6f})")
else:
    # ── Option B: No API key — print prompt to paste manually ────────────
    print("=" * 65)
    print("📢 NO API KEY DETECTED — COPY & PASTE PROMPT INTO ANY LLM CHAT WINDOW")
    print("=" * 65)
    print("You can test this RAG prompt for FREE! Copy everything between the")
    print("dashed lines and paste it into ChatGPT (chatgpt.com), Claude (claude.ai),")
    print("Gemini (gemini.google.com), or your favorite LLM chat window:\n")
    print("------------------------- 📋 CLIPBOARD COPY START -------------------------")
    print(f"System Instruction:\n{SYSTEM_PROMPT}\n")
    print(USER_PROMPT)
    print("-------------------------- 📋 CLIPBOARD COPY END --------------------------")
    print("=" * 65)


---
## Experiment: Try Different Questions

Try asking different questions about the text and watch how the retrieval dynamically finds the exact relevant chunks!


In [ ]:
# ── Quick helper to run the full pipeline in one cell ─────────────────────

def ask(question, k=3):
    # 1. Embed question
    q_vec = embed_model.encode(question).tolist()

    # 2. Retrieve top-k chunks
    res = collection.query(query_embeddings=[q_vec], n_results=k,
                           include=["documents", "distances"])
    docs = res["documents"][0]
    scores = [round(1 - d, 4) for d in res["distances"][0]]

    # 3. Show retrieved chunks
    print(f"\nQuestion: {question}")
    print(f"Retrieved {k} chunks (similarity scores: {scores})")

    # 4. Augment
    context = "\n\n".join(f"[{i+1}] {t}" for i, t in enumerate(docs))
    user_msg = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"

    # 5. Generate (if API key available)
    if openai_key:
        from openai import OpenAI
        r = OpenAI(api_key=openai_key).chat.completions.create(
            model="gpt-4o-mini", temperature=0, max_tokens=300,
            messages=[
                {"role": "system", "content": "Answer using only the context provided. If not present, state so."},
                {"role": "user", "content": user_msg}
            ]
        )
        print(f"\nAnswer: {r.choices[0].message.content}")
    else:
        print("\n📋 PROMPT READY TO PASTE INTO ANY LLM CHAT WINDOW:")
        print("┌" + "─" * 64 + "┐")
        for line in user_msg.splitlines():
            print(f"  {line}")
        print("└" + "─" * 64 + "┘")
        print("💡 Paste this into ChatGPT, Claude, or Gemini to see the answer!")

    print("-" * 60)


# ── Test with curated questions ───────────────────────────────────────────
ask("Where can water be found on Mars today?")
ask("What did the Curiosity rover discover?")
ask("What is Olympus Mons?")
ask("What is the diameter of Jupiter's Great Red Spot?")  # Negative test: not in context


---
## What You Learned — Core RAG Takeaways

### The Core RAG Workflow:

1. **Chunking**: Slicing large text documents into small, coherent pieces with a slight overlap.
2. **Embedding**: Converting text into dense semantic vector representations where similar meanings have high cosine similarity.
3. **Vector Database**: Indexing vectors and metadata for sub-millisecond retrieval (e.g. ChromaDB, Pinecone, Qdrant, Milvus).
4. **Retrieval**: Finding the top-K closest chunks based on cosine distance.
5. **Prompt Augmentation**: Feeding the retrieved chunks into an LLM prompt as grounded context.
6. **Generation**: Generating an accurate, factual response without hallucinations.

Whether you run on your laptop with ChromaDB or in the cloud at scale, this fundamental pipeline remains identical!
